# Longest Common Subsequence (LCS)

## Introduction

A **subsequence** of a string is obtained by deleting zero or more characters without changing the order of the remaining characters. For example, `"ACE"` is a subsequence of `"ABCDE"`, but `"AEC"` is not.

The **Longest Common Subsequence (LCS)** problem asks: given two strings `A` and `B`, what is the longest string that is a subsequence of both?

Unlike a *substring*, the characters of a subsequence do **not** have to be contiguous.

### Approach: Dynamic Programming

Brute force would check all 2ⁿ subsequences of one string against the other, which is infeasible. LCS has two properties that make dynamic programming a good fit:

1. **Optimal substructure:** the LCS of two strings is built from the LCS of their shorter prefixes.
2. **Overlapping subproblems:** the same prefix pairs are needed repeatedly, so we compute each once and store it in a table.

### Notebook outline

1. Import libraries
2. Implement `LCS(A, B)`: build the DP table, then trace back to recover the actual subsequence
3. Test on a classic example
4. Analyse the result and the complexity

## Step 1: Import Libraries

We use **NumPy** to create the 2-D table that stores the LCS lengths for every pair of prefixes. A plain list of lists would work equally well, but `np.zeros` gives a convenient pre-filled grid.

In [1]:
import numpy as np

## Step 2: The `LCS(A, B)` Function

The function works in two phases: **fill the table**, then **trace back**.

### Phase 1: Build the DP table

Let `n = len(A)` and `m = len(B)`. We create a table `c` of size `(n+1) × (m+1)`, initialised to zeros.

> `c[i][j]` = length of the LCS of the first `i` characters of `A` and the first `j` characters of `B`.

Row `0` and column `0` stay at `0`, because an LCS with an empty string is always empty.

For every `i` from `1..n` and `j` from `1..m`, we apply the recurrence:

- **If `A[i-1] == B[j-1]`:** the characters match, so we extend the LCS of the two shorter prefixes:
  `c[i][j] = c[i-1][j-1] + 1`
- **Otherwise:** we drop the last character of either `A` or `B` and keep the better result:
  `c[i][j] = max(c[i-1][j], c[i][j-1])`

(`A[i-1]` and `B[j-1]` are used because the table is 1-indexed while Python strings are 0-indexed.)

After the loops finish, `c[n][m]` holds the **length** of the LCS.

### Phase 2: Traceback to recover the subsequence

The table only gives the length, so we walk backwards from `c[n][m]` to rebuild the string. Starting at `(i, j) = (n, m)`:

- **Characters match:** this character belongs to the LCS. Prepend it to the result `x` and move diagonally (`i -= 1`, `j -= 1`).
- **Characters differ:** move in the direction of the larger neighbouring value.
  - If `c[i-1][j] >= c[i][j-1]`, move up (`i -= 1`).
  - Otherwise, move left (`j -= 1`).

The walk ends when either index reaches `0`. Since we walk from the end to the start, each matched character is **prepended** to `x`.

> **Note:** A pair of strings can have several different LCSs of the same length. The `>=` tie-break decides which one is returned: it prefers moving up when both neighbours are equal.

In [2]:
def LCS(A,B):
    n = len(A)
    m = len(B)
    c = np.zeros((n+1,m+1))
    for i in range(1,n+1):
        for j in range(1,m+1):
            if(A[i-1]==B[j-1]):
                c[i,j] = c[i-1,j-1]+1
            else:
                c[i,j] = max(c[i-1,j],c[i,j-1])
    i = n
    j = m
    x =""
    while(i > 0 and j >0):
        if(A[i-1] == B[j-1]):
            x = A[i-1] + x  
            i-=1
            j-=1
        else:
            if(c[i-1][j] >= c[i][j-1]):
                i-=1
            else:
                j-=1
    return x

## Step 3: Testing the Function

We use a standard textbook example (from *Introduction to Algorithms*, CLRS):

- `A = "ABCBDAB"` (length 7)
- `B = "BDCABA"` (length 6)

The expected LCS length is **4**. Several valid answers exist, including `BCBA`, `BDAB` and `BCAB`.

In [3]:
A = "ABCBDAB"
B = "BDCABA"

s = LCS(A,B)

print(s)

BCBA


## Result

**Output:** `BCBA`

This is a valid LCS of length 4: it appears in order in both strings.

| String | Characters of `BCBA` highlighted |
|---|---|
| `A = ABCBDAB` | A **B** **C** **B** D **A** B |
| `B = BDCABA` | **B** D **C** A **B** **A** |

### DP table for this example

Rows are prefixes of `A`, columns are prefixes of `B`.

| | ∅ | B | D | C | A | B | A |
|---|---|---|---|---|---|---|---|
| **∅** | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| **A** | 0 | 0 | 0 | 0 | 1 | 1 | 1 |
| **B** | 0 | 1 | 1 | 1 | 1 | 2 | 2 |
| **C** | 0 | 1 | 1 | 2 | 2 | 2 | 2 |
| **B** | 0 | 1 | 1 | 2 | 2 | 3 | 3 |
| **D** | 0 | 1 | 2 | 2 | 2 | 3 | 3 |
| **A** | 0 | 1 | 2 | 2 | 3 | 3 | 4 |
| **B** | 0 | 1 | 2 | 2 | 3 | 4 | 4 |

The bottom-right cell is `4`, which confirms the LCS length.

### Traceback walkthrough

Starting at `(7, 6)`:

1. `(7,6)`: `B ≠ A`, up is not smaller than left → move **up**
2. `(6,6)`: `A = A` → take **A**, move diagonally
3. `(5,5)`: `D ≠ B`, up (3) ≥ left (2) → move **up**
4. `(4,5)`: `B = B` → take **B**, move diagonally
5. `(3,4)`: `C ≠ A`, left (2) > up (1) → move **left**
6. `(3,3)`: `C = C` → take **C**, move diagonally
7. `(2,2)`: `B ≠ D`, left (1) > up (0) → move **left**
8. `(2,1)`: `B = B` → take **B**, move diagonally → reached the edge

Collected in reverse and prepended each time, this gives **`BCBA`**.

### Observations

- A different tie-break rule (preferring left over up) could return another valid LCS such as `BDAB`.
- `np.zeros` creates a float array by default, so the table holds values like `4.0`. This doesn't affect the result, but `dtype=int` would be cleaner.

## Time and Space Complexity

Let `n = len(A)`, `m = len(B)`.

### Time Complexity: **O(n × m)**

| Phase | Cost |
|---|---|
| Filling the DP table | Two nested loops, with O(1) work per cell → **O(n·m)** |
| Traceback | Each step decreases `i`, `j` or both → at most **O(n + m)** steps |
| Building the result string | Prepending to a string costs O(L) per match, so at most O(L²) with L ≤ min(n, m), which is still bounded by O(n·m) |

**Overall: O(n × m).**

### Space Complexity: **O(n × m)**

- The table `c` has `(n+1) × (m+1)` cells → **O(n·m)**
- The output string takes O(min(n, m)) extra, which is dominated by the table.

### Possible optimisations

- **Length only:** if only the length is needed, keep two rows (or one) instead of the full table → **O(min(n, m))** space.
- **Recovering the sequence in linear space:** *Hirschberg's algorithm* returns the actual LCS using **O(n + m)** space, with the same O(n·m) time.

### Comparison with brute force

| Approach | Time |
|---|---|
| Brute force (check all subsequences of `A`) | O(2ⁿ · m) |
| Dynamic programming (this notebook) | **O(n · m)** |